# Neural Network xG Model

In [ ]:
import pandas as pd
import football_analytics.utils.helper as helper
import football_analytics.utils.visuals as visuals
from sklearn.metrics import log_loss
from importlib import reload
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import joblib
reload(helper)
reload(visuals)

<module 'football_analytics.utils.visuals' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\utils\\visuals.py'>

### Load Supabase Data

In [201]:
table_name = 'shots'
key_column = 'statsbomb_event_id'

In [4]:
data = helper.fetch_all_rows_in_batches(table_name=table_name, key_column=key_column, batch_size=5000)

Fetched 5000 rows (total 5000).
Fetched 5000 rows (total 10000).
Fetched 5000 rows (total 15000).
Fetched 5000 rows (total 20000).
Fetched 5000 rows (total 25000).
Fetched 5000 rows (total 30000).
Fetched 5000 rows (total 35000).
Fetched 5000 rows (total 40000).
Fetched 5000 rows (total 45000).
Fetched 5000 rows (total 50000).
Fetched 5000 rows (total 55000).
Fetched 5000 rows (total 60000).
Fetched 5000 rows (total 65000).
Fetched 5000 rows (total 70000).
Fetched 5000 rows (total 75000).
Fetched 5000 rows (total 80000).
Fetched 5000 rows (total 85000).
Fetched 3023 rows (total 88023).


In [202]:
df_raw = pd.DataFrame(data)

## Prepare Data

In [203]:
df = df_raw.dropna().copy()

In [204]:
df['goal_outcome'] = df['outcome'].apply(lambda x: 1 if x=='Goal' else 0)
df['is_with_feet'] = df['body_part'].apply(lambda x: 1 if (x=='Right Foot' or x=='Left Foot') else 0)
df['is_penalty'] = df['shot_type'].apply(lambda x: 1 if x=='Penalty' else 0)

In [205]:
y = df['goal_outcome'].values.astype(int)
X = np.column_stack([
    df['distance_to_goal'].values, 
    df['angle_to_goal_deg'].values, 
    df['keeper_distance'].values, 
    df['min_defender_distance'].values,
    df['avg_defender_distance'].values,
    df['is_penalty'].values,
    df['num_def_in_shot_triangle'].values,
    df['frac_goal_uncovered'].values,
                     ])

#### Check for NaN and Inf in Input / Output

In [206]:
if np.isnan(y).sum() > 0:
    raise ValueError("NaN values found in target variable y")
if np.isnan(X).sum() > 0:
    raise ValueError("NaN values found in feature matrix X")

## Define Loader, Model and Optimizer

In [207]:
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

In [208]:
class ShotDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [209]:
dataset = ShotDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [210]:
import torch.nn as nn

class XGModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # output = logit
        )

    def forward(self, x):
        return self.net(x)


In [211]:
model = XGModel(input_dim=X.shape[1])

In [212]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [213]:
np.isnan(X).sum(), np.isinf(X).sum()

(np.int64(0), np.int64(0))

In [214]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss / len(loader):.4f}")


Epoch 01 | Loss: 0.2983
Epoch 02 | Loss: 0.2734
Epoch 03 | Loss: 0.2721
Epoch 04 | Loss: 0.2715
Epoch 05 | Loss: 0.2708
Epoch 06 | Loss: 0.2703
Epoch 07 | Loss: 0.2697
Epoch 08 | Loss: 0.2694
Epoch 09 | Loss: 0.2691
Epoch 10 | Loss: 0.2690
Epoch 11 | Loss: 0.2689
Epoch 12 | Loss: 0.2686
Epoch 13 | Loss: 0.2687
Epoch 14 | Loss: 0.2686
Epoch 15 | Loss: 0.2683
Epoch 16 | Loss: 0.2684
Epoch 17 | Loss: 0.2682
Epoch 18 | Loss: 0.2679
Epoch 19 | Loss: 0.2678
Epoch 20 | Loss: 0.2679
Epoch 21 | Loss: 0.2677
Epoch 22 | Loss: 0.2676
Epoch 23 | Loss: 0.2675
Epoch 24 | Loss: 0.2674
Epoch 25 | Loss: 0.2675
Epoch 26 | Loss: 0.2673
Epoch 27 | Loss: 0.2673
Epoch 28 | Loss: 0.2674
Epoch 29 | Loss: 0.2672
Epoch 30 | Loss: 0.2669


In [218]:
def xg_log_loss(y_true, y_pred):
    """
    Computes binary cross-entropy (log loss) for xG predictions.
    """
    return log_loss(y_true, y_pred)

In [219]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X, dtype=torch.float32))
    xg_probs = torch.sigmoid(logits).numpy().flatten()


In [220]:
xg_log_loss_nn = xg_log_loss(y, xg_probs)
print(f"NN xG log loss: {xg_log_loss_nn:.4f}")

NN xG log loss: 0.2665


## Save Model

In [ ]:
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f"nn_xg_model_{date_str}.pth"
torch.save(model.state_dict(), f'../nn_models/{model_name}')
joblib.dump(scaler, f'../nn_models/scaler_{date_str}.save')

In [222]:
df['nn_xg'] = xg_probs

In [223]:
df[['nn_xg', 'statsbomb_xg','statsbomb_event_id']].to_csv('nn_xg_model_predictions_2.csv', index=False)

## Check Differences

In [247]:
event_id = '18430765-a2d8-4ffa-94b2-d0dc6184dbe3'

In [248]:
shot = df[df['statsbomb_event_id'] == event_id]

In [249]:
shot['outcome']

8432    Goal
Name: outcome, dtype: object

In [250]:
X_tmp = np.column_stack([
    shot['distance_to_goal'].values, 
    shot['angle_to_goal_deg'].values, 
    shot['keeper_distance'].values, 
    shot['min_defender_distance'].values,
    shot['avg_defender_distance'].values,
    shot['is_penalty'].values,
    shot['num_def_in_shot_triangle'].values,
    shot['frac_goal_uncovered'].values,
                     ])

In [251]:
X_tmp

array([[ 8.54400375, 48.31094151, 18.97366596,  1.        ,  6.50292321,
         0.        ,  1.        ,  0.55284398]])

In [252]:
X_tmp = scaler.transform(X_tmp)

In [253]:
X_tmp

array([[-1.21590398,  1.4501137 ,  0.31136026, -0.87517369, -1.21530544,
        -0.12532415,  0.21036086,  0.01875552]])

In [254]:
with torch.no_grad():
    logits = model(torch.tensor(X_tmp, dtype=torch.float32))
    xg_tmp = torch.sigmoid(logits).numpy().flatten()

In [255]:
xg_tmp

array([0.01439589], dtype=float32)

In [256]:
shot_dict = helper.parse_json_field(shot.iloc[0]['full_json'])

In [257]:
visuals.plot_shot_details(shot_dict)